In [1]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/AI Resume Screening"):
    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{size_mb:8.2f} MB  {path}")

In [2]:
!pip uninstall -y transformers sentence-transformers pillow pdfplumber

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
ERROR: Exception:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/shutil.py", line 847, in move
    os.rename(src, real_dst)
FileNotFoundError: [Errno 2] No such file or directory: '/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/bin/transformers' -> '/private/var/folders/d5/_xxsp4_d0716jz7wj0717pkr0000gn/T/pip-uninstall-23oh57l6/transformers'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 110, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/li

In [3]:
!pip install --no-cache-dir \
transformers==4.55.4 \
sentence-transformers==5.1.0 \
pillow==12.3.0 \
pdfplumber==0.11.10 \
python-docx \
faiss-cpu \
gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 520.8 kB/s  0:00:21m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 794.2 kB/s  0:00:06 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sentence-transformers]ence-transformers]


In [4]:
import os
import torch
import faiss
import gradio as gr
import pdfplumber
import docx
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

/Users/dipeshkardate/Desktop/AI-Powered-Resume-Screening-And-Job-Matching-System/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"

# DATA_PATH = BASE_PATH
# MODEL_PATH = BASE_PATH + "models/"
# OUTPUT_PATH = BASE_PATH + "outputs/"

In [ ]:
BASE_PATH = "/content/drive/MyDrive/AI Resume Screening/"

DATA_PATH = BASE_PATH + "data/"
MODEL_PATH = BASE_PATH + "models/"
OUTPUT_PATH = BASE_PATH + "output/"

In [ ]:
job_df = pd.read_csv(DATA_PATH + "processed_jobs.csv")

index = faiss.read_index(MODEL_PATH + "job_faiss.index")

print("Jobs:", len(job_df))
print("Index:", index.ntotal)

Jobs: 1615940
Index: 1615940


In [ ]:
with open(MODEL_PATH + "embedding_model.txt") as f:
    MODEL_NAME = f.read().strip()

EMBEDDING_MODEL = SentenceTransformer(MODEL_NAME)

print("Embedding model:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: BAAI/bge-small-en-v1.5


In [ ]:
CROSS_ENCODER_PATH = MODEL_PATH + "resume_matching_model"

tokenizer = AutoTokenizer.from_pretrained(CROSS_ENCODER_PATH)

cross_encoder = AutoModelForSequenceClassification.from_pretrained(
    CROSS_ENCODER_PATH
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cross_encoder.to(device)
cross_encoder.eval()

print("Device:", device)

Device: cpu


In [ ]:
def extract_resume_text(file):

    file_path = file.name if hasattr(file, "name") else file

    if file_path.endswith(".pdf"):

        text = ""

        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

        return text

    elif file_path.endswith(".docx"):

        document = docx.Document(file_path)

        return "\n".join(
            paragraph.text
            for paragraph in document.paragraphs
        )

    elif file_path.endswith(".txt"):

        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    else:
        raise ValueError("Unsupported file format.")

In [ ]:
def get_resume_embedding(resume_text):

    embedding = EMBEDDING_MODEL.encode(
      resume_text,
      convert_to_numpy=True,
      normalize_embeddings=True,
      device="cuda" if torch.cuda.is_available() else "cpu"
    )

    return embedding.astype("float32")

In [ ]:
TOP_K = 50

def retrieve_jobs(resume_embedding):

    scores, indices = index.search(

        resume_embedding.reshape(1, -1),

        TOP_K

    )

    jobs = job_df.iloc[indices[0]].copy()

    jobs["faiss_score"] = scores[0]

    return jobs.reset_index(drop=True)

In [ ]:
import torch

device = next(cross_encoder.parameters()).device

def rerank_jobs(resume_text, jobs_df):

    cross_encoder.eval()

    pairs = [
        (resume_text, row["job_text"])
        for _, row in jobs_df.iterrows()
    ]

    inputs = tokenizer(
        [p[0] for p in pairs],
        [p[1] for p in pairs],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = cross_encoder(**inputs)
        scores = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

    jobs_df = jobs_df.copy()
    jobs_df["cross_encoder_score"] = scores

    return jobs_df.sort_values(
        by="cross_encoder_score",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
skill_list = [

    "python",
    "java",
    "c++",
    "sql",
    "excel",
    "power bi",
    "tableau",
    "aws",
    "azure",
    "gcp",
    "docker",
    "kubernetes",
    "git",
    "github",

    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "keras",
    "xgboost",
    "lightgbm",

    "langchain",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "nlp",
    "computer vision",
    "transformers",
    "bert",
    "llm",
    "langchain",
    "langgraph",
    "rag",
    "faiss",

    "statistics",
    "data analysis",
    "data visualization",
    "feature engineering",

    "communication",
    "leadership",
    "teamwork",
    "problem solving",
    "customer service",
    "sales",
    "marketing",
    "human resources",
    "recruitment",
    "hr"
]

In [ ]:
def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in skill_list:
        if skill in text:
            found_skills.append(skill)

    return list(set(found_skills))

In [ ]:
def compare_skills(resume_skills, job_skills):

    matched_skills = list(
        set(resume_skills).intersection(set(job_skills))
    )

    missing_skills = list(
        set(job_skills) - set(resume_skills)
    )

    if len(job_skills) == 0:
        score = 0.0
    else:
        score = (
            len(matched_skills) / len(job_skills)
        ) * 100

    return {
        "matched_skills": sorted(matched_skills),
        "missing_skills": sorted(missing_skills),
        "skill_match_score": round(score, 2)
    }

In [ ]:
def explain_match(resume_text, job):

    resume_skills = extract_skills(resume_text)

    job_skills = extract_skills(job["job_text"])

    comparison = compare_skills(
        resume_skills,
        job_skills
    )

    return {
        "Job Title": job["Job Title"],
        "Company": job["Company"],
        "Cross Encoder Score": round(job["cross_encoder_score"], 3),
        "FAISS Score": round(job["faiss_score"], 3),
        "Matched Skills": comparison["matched_skills"],
        "Missing Skills": comparison["missing_skills"],
        "Skill Match (%)": round(comparison["skill_match_score"], 2)
    }

In [ ]:
import pandas as pd

def predict_jobs(file):

    try:

        # Extract resume text
        resume_text = extract_resume_text(file)

        if not resume_text.strip():
            raise ValueError("Resume text extraction failed.")

        # Generate embedding
        resume_embedding = get_resume_embedding(resume_text)

        # Retrieve jobs using FAISS
        retrieved_jobs = retrieve_jobs(resume_embedding)

        # Re-rank using Cross Encoder
        ranked_jobs = rerank_jobs(
            resume_text,
            retrieved_jobs
        )

        reports = []

        for _, job in ranked_jobs.head(10).iterrows():

            report = explain_match(
                resume_text,
                job
            )

            reports.append({
                "Job Title": report["Job Title"],
                "Company": report["Company"],
                "Match Score": report["Cross Encoder Score"],
                "Skill Match (%)": report["Skill Match (%)"],
                "Matched Skills": ", ".join(report["Matched Skills"])
                if report["Matched Skills"] else "None",

                "Missing Skills": ", ".join(report["Missing Skills"])
                if report["Missing Skills"] else "None"
            })

        results = pd.DataFrame(reports)

        return (
            resume_text[:1500],
            results,
            "✅ Successfully matched jobs."
        )

    except Exception as e:

        return (
            "",
            pd.DataFrame(),
            f"❌ {str(e)}"
        )

In [ ]:
import gradio as gr
print(gr.__version__)

6.17.3


In [ ]:
import gradio as gr

demo = gr.Interface(

    fn=predict_jobs,

    inputs=gr.File(
        label="📄 Upload Resume (PDF / DOCX / TXT)",
        type="filepath"
    ),

    outputs=[
        gr.Textbox(
            label="📋 Resume Preview",
            lines=12
        ),

        gr.Dataframe(
            headers=[
                "Job Title",
                "Company",
                "Match Score",
                "Skill Match (%)",
                "Matched Skills",
                "Missing Skills"
            ],
            label="🎯 Top Job Recommendations"
        ),

        gr.Textbox(
            label="Status"
        )
    ],

    title="🤖 AI Resume Screening & Job Matching System",

    description="""
Upload a Resume (PDF, DOCX or TXT).

The system will:

• Extract Resume Text

• Generate BGE Embeddings

• Retrieve Jobs using FAISS

• Re-rank using Cross Encoder

• Explain Skill Matching
"""
)

In [ ]:
demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec7a30372aa200e8a5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
